# Section 1: Extract US Politics Sequences from WikiText-103

In [1]:
import hashlib, re
from datasets import load_dataset, load_from_disk
from tqdm.notebook import tqdm
from transformers import AutoTokenizer

NUM_SEQUENCES   = 100
NUM_PARAPHRASES = 10   # M=10; original is paraphrase 0, so 9 new paraphrases generated
SEQ_LEN         = 100  # exact token length for every row
SEED            = 42
SAVE_PATH       = "wikitext_prefix_paraphrase"
CONTROL_SAVE_PATH = "wikitext_prefix_paraphrase_control"
NUM_SUFFIX_SEQUENCES = NUM_SEQUENCES * NUM_PARAPHRASES  # 1000

In [2]:
_US_POLITICS_ANCHORS = [
    "congress", "senate", "republican", "democrat", "white house",
    "house of representatives", "obama", "trump", "biden", "clinton",
    "gop", "democratic party", "u.s. president", "u.s. congress",
    "u.s. senate", "midterm", "filibuster", "oval office",
]

_US_POLITICS_KEYWORDS = [
    "election", "president", "legislation", "political party",
    "congressman", "senator", "governor", "ballot", "partisan",
    "bipartisan", "caucus", "campaign", "polling", "primaries",
    "inauguration", "executive order", "speaker of the house",
]


def _is_us_politics(text: str) -> bool:
    """Loose filter for suffix/control pools: >= 2 anchors AND >= 4 total."""
    lower = text.lower()
    anchor_hits = sum(1 for kw in _US_POLITICS_ANCHORS if kw in lower)
    if anchor_hits < 2:
        return False
    support_hits = sum(1 for kw in _US_POLITICS_KEYWORDS if kw in lower)
    return (anchor_hits + support_hits) >= 4


def _is_us_politics_strict(text: str) -> bool:
    """Strict filter matching paraphrase_dataset.ipynb: >= 3 anchors AND >= 6 total."""
    lower = text.lower()
    anchor_hits = sum(1 for kw in _US_POLITICS_ANCHORS if kw in lower)
    if anchor_hits < 3:
        return False
    support_hits = sum(1 for kw in _US_POLITICS_KEYWORDS if kw in lower)
    return (anchor_hits + support_hits) >= 6

In [3]:
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
dataset   = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")
no_headers = dataset.filter(lambda x: not re.match(r"(?: =)+[^=]+(?:= )+", x["text"]), desc="Filtering headers", num_proc=16)

# ── Prefix pool: replicate paraphrase_dataset.ipynb filtering exactly ──
strict = no_headers.filter(lambda x: _is_us_politics_strict(x["text"]), desc="Strict topic filter (prefix pool)", num_proc=16)
strict = strict.filter(lambda x: len(tokenizer(x["text"])["input_ids"]) > (SEQ_LEN + 50), desc="Min length filter (prefix pool)", num_proc=16)
strict = strict.shuffle(seed=SEED)
prefix_pool = list(strict)[:NUM_SEQUENCES]
print(f"Prefix sources (strict filter): {len(prefix_pool)} (from {len(strict)} candidates)")

# ── Suffix/control pools: looser filtering, excluding prefix texts ──
loose = no_headers.filter(lambda x: _is_us_politics(x["text"]), desc="Loose topic filter (suffix/control)", num_proc=16)
loose = loose.filter(lambda x: len(tokenizer(x["text"])["input_ids"]) > 150, desc="Min length filter (suffix/control)", num_proc=16)
loose = loose.shuffle(seed=SEED)

prefix_texts = {p["text"] for p in prefix_pool}
remaining = [ex for ex in loose if ex["text"] not in prefix_texts]

needed = NUM_SUFFIX_SEQUENCES + NUM_SUFFIX_SEQUENCES  # 500 suffix + 500 control
print(f"Remaining after excluding prefix texts: {len(remaining)} (need {needed})")
assert len(remaining) >= needed, f"Not enough sequences: got {len(remaining)}, need {needed}"

suffix_pool  = remaining[:NUM_SUFFIX_SEQUENCES]
control_pool = remaining[NUM_SUFFIX_SEQUENCES:]

print(f"Suffix sources: {len(suffix_pool)}")
print(f"Control pool:   {len(control_pool)}")

Prefix sources (strict filter): 100 (from 656 candidates)
Remaining after excluding prefix texts: 2138 (need 2000)
Suffix sources: 1000
Control pool:   1138


# Section 2: Extract Prefixes (first 50 tokens)

In [4]:
PREFIX_TOKENS = 50

def extract_prefix(tokens, tokenizer):
    """Take exactly 50 tokens as prefix."""
    assert len(tokens) >= PREFIX_TOKENS, f"Paraphrase too short: {len(tokens)} tokens"
    prefix = tokens[:PREFIX_TOKENS]
    return prefix, tokenizer.decode(prefix)


prefix_sources = []
for i, example in enumerate(tqdm(prefix_pool, desc="Extracting prefixes")):
    tokens = tokenizer.encode(example["text"])
    prefix_tokens, prefix_text = extract_prefix(tokens, tokenizer)
    prefix_sources.append({
        "seq_idx":       i,
        "prefix_tokens": prefix_tokens,
        "prefix_text":   prefix_text,
        "full_text":     example["text"],
    })
    print(f"  [{i}] {len(prefix_tokens)} tokens: {prefix_text[:80]}...")

lengths = [len(s["prefix_tokens"]) for s in prefix_sources]
print(f"\nPrefix token lengths: min={min(lengths)}, max={max(lengths)}, mean={sum(lengths)/len(lengths):.1f}")

Extracting prefixes:   0%|          | 0/100 [00:00<?, ?it/s]

  [0] 50 tokens:  The voters returned Sherman to office for a third term in 1858 . After a brief ...
  [1] 50 tokens:  The Democratic @-@ Republican congressional caucus convened February 25 , 1804 ...
  [2] 50 tokens:  Garfield during this time purchased the property in Mentor that reporters later...
  [3] 50 tokens:  On September 7 , Paul called for a vote of no confidence in U.S. Treasury Secre...
  [4] 50 tokens:  The acts became controversial from prosecution thereunder of a Congressman and ...
  [5] 50 tokens:  Black was chosen as the Democratic nominee for lieutenant governor in 1915 , de...
  [6] 50 tokens:  When New York 's long @-@ serving U.S. Senator Daniel Patrick Moynihan announce...
  [7] 50 tokens:  On March 2 , Paul was one of nine senators to vote against a stopgap bill that ...
  [8] 50 tokens:  In 1857 – 1858 , Douglas broke with President James Buchanan , leading to a fig...
  [9] 50 tokens:  Franklin County , which has elected Kilroy twice , makes up 87 % of the 1

# Section 3: Paraphrase Prefixes with OpenAI API

In [5]:
import json, os, time
from openai import OpenAI
from dotenv import load_dotenv

OPENAI_MODEL = "gpt-5-mini-2025-08-07"
CACHE_PATH   = "paraphrase_cache.json"

load_dotenv()
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

In [6]:
def paraphrase(text, n=NUM_PARAPHRASES - 1):
    """Generate n paraphrases of full text (reused from paraphrase_dataset.ipynb as fallback)."""
    prompt = (
        f"Paraphrase the following text {n} times. "
        f"Write paraphrases of similar length (at least 100 words). "
        f"Return ONLY a JSON object with key 'paraphrases' whose value is an array of {n} strings. "
        f"\n\nText: {text}"
    )
    for attempt in range(5):
        try:
            resp = client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[{"role": "user", "content": prompt}],
                response_format={"type": "json_object"},
            )
            result = json.loads(resp.choices[0].message.content)["paraphrases"]
            if len(result) != n:
                raise ValueError(f"Expected {n} paraphrases but got {len(result)}")
            return result
        except Exception as e:
            print(f"  Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)
    raise RuntimeError("Failed after 5 retries")

In [7]:
from concurrent.futures import ThreadPoolExecutor, as_completed

if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH) as f:
        cache = json.load(f)
else:
    cache = {}

# Identify cache misses
to_fetch = []  # list of (seq, cache_key)
for seq in prefix_sources:
    key = hashlib.sha256(f"{seq['full_text']}|{OPENAI_MODEL}".encode()).hexdigest()
    seq["cache_key"] = key

    expected = NUM_PARAPHRASES - 1
    if key in cache and len(cache[key]) < expected:
        print(f"Cache entry {key[:8]}… has {len(cache[key])} paraphrases (expected {expected}), regenerating.")
        del cache[key]

    if key not in cache:
        to_fetch.append((seq, key))

print(f"Cache hits: {len(prefix_sources) - len(to_fetch)}, misses: {len(to_fetch)}")

# Fetch all misses in parallel
failed_keys = set()
if to_fetch:
    MAX_WORKERS = 16

    def _fetch(item):
        seq, key = item
        return key, seq["seq_idx"], paraphrase(seq["full_text"])

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_fetch, item): item[1] for item in to_fetch}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Fetching paraphrases"):
            key = futures[future]
            try:
                key, seq_idx, result = future.result()
                cache[key] = result
            except Exception as e:
                print(f"  SKIP key={key[:8]}…: {e}")
                failed_keys.add(key)

    # Save cache once after all fetches
    with open(CACHE_PATH, "w") as f:
        json.dump(cache, f, indent=2)
    print(f"Saved {len(to_fetch) - len(failed_keys)} new entries to cache, skipped {len(failed_keys)}.")

# Remove failed sequences from prefix_sources
if failed_keys:
    prefix_sources = [seq for seq in prefix_sources if seq["cache_key"] not in failed_keys]
    print(f"Kept {len(prefix_sources)} prefix sources after skipping failures.")

print(f"All {len(prefix_sources)} prefix sources have paraphrases.")

Cache hits: 99, misses: 1


Fetching paraphrases:   0%|          | 0/1 [00:00<?, ?it/s]

  Attempt 1 failed: Expected 9 paraphrases but got 8
  Attempt 2 failed: Expected 9 paraphrases but got 8
Saved 1 new entries to cache, skipped 0.
All 100 prefix sources have paraphrases.


# Section 4: Collect Suffix Sequences

In [8]:
suffix_sequences = []
for example in tqdm(suffix_pool, desc="Collecting suffix sequences"):
    tokens = tokenizer.encode(example["text"])
    suffix_sequences.append({"tokens": tokens, "text": example["text"]})

print(f"Collected {len(suffix_sequences)} suffix sequences")
assert len(suffix_sequences) == NUM_SUFFIX_SEQUENCES

Collected 1000 suffix sequences


# Section 5: Assemble Paraphrase Dataset (N×M rows, 100 tokens each)

In [9]:
rows = []
suffix_idx = 0
actual_num_sequences = len(prefix_sources)
actual_total = actual_num_sequences * NUM_PARAPHRASES

for seq in tqdm(prefix_sources, desc="Assembling dataset"):
    paraphrases = cache[seq["cache_key"]]

    for para_ver in range(NUM_PARAPHRASES):
        if para_ver == 0:
            prefix_ids = list(seq["prefix_tokens"])
        else:
            # Extract prefix from the full-text paraphrase
            para_text = paraphrases[para_ver - 1]
            para_tokens = tokenizer.encode(para_text)
            prefix_ids, _ = extract_prefix(para_tokens, tokenizer)
            prefix_ids = list(prefix_ids)

        suffix_ids = suffix_sequences[suffix_idx]["tokens"]

        # Single truncation on the combined sequence
        combined = (prefix_ids + suffix_ids)[:SEQ_LEN]

        if len(combined) < SEQ_LEN:
            print(f"  WARNING: source_idx={seq['seq_idx']} para={para_ver} "
                  f"combined only {len(combined)} tokens (prefix={len(prefix_ids)}, suffix={len(suffix_ids)})")
            combined = combined + [tokenizer.eos_token_id] * (SEQ_LEN - len(combined))

        rows.append({
            "input_ids":      combined,
            "attention_mask": [1] * SEQ_LEN,
            "text":           tokenizer.decode(combined),
            "source_idx":     seq["seq_idx"],
            "paraphrase_idx": para_ver,
        })
        suffix_idx += 1

from datasets import Dataset
ds = Dataset.from_list(rows)
ds.save_to_disk(SAVE_PATH)
print(f"Saved {len(ds)} rows to {SAVE_PATH}/")
print(f"Target: {actual_total} rows ({actual_num_sequences} sequences × {NUM_PARAPHRASES} paraphrases)")

# Report prefix token length distribution
prefix_lens = []
for seq in prefix_sources:
    prefix_lens.append(len(seq["prefix_tokens"]))
    for para_text in cache[seq["cache_key"]]:
        para_tokens = tokenizer.encode(para_text)
        p_ids, _ = extract_prefix(para_tokens, tokenizer)
        prefix_lens.append(len(p_ids))
print(f"Prefix token lengths across all versions: min={min(prefix_lens)}, max={max(prefix_lens)}, mean={sum(prefix_lens)/len(prefix_lens):.1f}")

Assembling dataset:   0%|          | 0/100 [00:00<?, ?it/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saved 1000 rows to wikitext_prefix_paraphrase/
Target: 1000 rows (100 sequences × 10 paraphrases)
Prefix token lengths across all versions: min=50, max=50, mean=50.0


In [10]:
from collections import defaultdict
from datasets import load_from_disk

ds_val = load_from_disk(SAVE_PATH)

# Check all input_ids have length 100
bad = [i for i, row in enumerate(ds_val) if len(row["input_ids"]) != SEQ_LEN]
assert not bad, f"Rows with wrong length: {bad[:5]}"
print(f"[PASS] All {len(ds_val)} rows have input_ids of length {SEQ_LEN}.")

groups = defaultdict(list)
for row in ds_val:
    groups[row["source_idx"]].append(row)

for src_idx, grp in groups.items():
    para_idxs = {r["paraphrase_idx"] for r in grp}
    assert 0 in para_idxs, f"source_idx {src_idx}: missing original (paraphrase_idx=0)"
    assert len(grp) == NUM_PARAPHRASES, f"source_idx {src_idx}: expected {NUM_PARAPHRASES} rows, got {len(grp)}"
print(f"[PASS] All {len(groups)} source groups contain the original and {NUM_PARAPHRASES} versions each.")
print(f"Total rows: {len(ds_val)} (expected: {actual_total})")

[PASS] All 1000 rows have input_ids of length 100.
[PASS] All 100 source groups contain the original and 10 versions each.
Total rows: 1000 (expected: 1000)


# Section 6: Control Dataset (500 independent WikiText sequences)

In [11]:
control_target = actual_total  # match the paraphrase dataset size

control_rows = []
for example in tqdm(control_pool, desc="Collecting control sequences"):
    if len(control_rows) >= control_target:
        break
    tokens = tokenizer.encode(example["text"])
    if len(tokens) < SEQ_LEN:
        continue
    control_rows.append({
        "input_ids":      tokens[:SEQ_LEN],
        "attention_mask": [1] * SEQ_LEN,
        "text":           example["text"],
    })

print(f"Collected {len(control_rows)} control rows (target: {control_target})")

ds_control = Dataset.from_list(control_rows[:control_target])
ds_control.save_to_disk(CONTROL_SAVE_PATH)
print(f"Saved {len(ds_control)} rows to {CONTROL_SAVE_PATH}/")

Collected 1000 control rows (target: 1000)


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Saved 1000 rows to wikitext_prefix_paraphrase_control/


In [12]:
ds_ctrl_val = load_from_disk(CONTROL_SAVE_PATH)

assert len(ds_ctrl_val) == control_target, \
    f"Expected {control_target} rows, got {len(ds_ctrl_val)}"
print(f"[PASS] Control dataset has {len(ds_ctrl_val)} rows.")

bad = [i for i, row in enumerate(ds_ctrl_val) if len(row["input_ids"]) != SEQ_LEN]
assert not bad, f"Rows with wrong length: {bad[:5]}"
print(f"[PASS] All rows have input_ids of length {SEQ_LEN}.")

[PASS] Control dataset has 1000 rows.
[PASS] All rows have input_ids of length 100.
